# Czyszczenie - warstwa SILVER

Milestone 2, zadanie 1: wczytanie pomiarów i pogody z bronze, odfiltrowanie anomalii wykrytych w Milestone 1 (notebook 03, sekcja 8) i zapis oczyszczonych danych do `data/silver/`.

Czyścimy wszystkie wskaźniki, nie tylko PM2.5: reguły ogólne dotyczą każdego pomiaru, a reguły szczegółowe tylko PM2.5, czyli zmiennej docelowej. Dzięki temu inne wskaźniki też są gotowe do późniejszego użycia.

Funkcje czyszczące są w `scripts/silver_clean.py`.

## 1. Wczytanie danych z bronze

In [1]:
from pathlib import Path
import runpy

import polars as pl

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
BRONZE_DIR = ROOT / 'data' / 'bronze'
SILVER_DIR = ROOT / 'data' / 'silver'

silver = runpy.run_path(str(ROOT / 'scripts' / 'silver_clean.py'))

measurements = pl.read_parquet(BRONZE_DIR / 'pomiary_2024.parquet')
weather = pl.read_parquet(BRONZE_DIR / 'openmeteo_2024.parquet')

print('pomiary:', measurements.shape)
print('pogoda: ', weather.shape)

pomiary: (8952726, 8)
pogoda:  (843264, 12)


## 2. Reguły czyszczenia

Każdy wiersz pomiarów dostaje kolumnę `powod_usuniecia` z pierwszą pasującą regułą albo `NULL`, jeśli zostaje. Kolejność reguł ma znaczenie: brak pomiaru sprawdzamy pierwszy, więc pusty duplikat liczy się jako brak pomiaru.

| Reguła | Dotyczy | Anomalia z M1 | Dlaczego usuwamy |
|---|---|---|---|
| brak pomiaru | wszystkie wskaźniki | 8.1 | `NULL` w `wartosc` nie niesie informacji |
| duplikat stanowiska | wszystkie | 8.2 | ten sam pomiar wyeksportowany w dwóch plikach; zostaje pierwsza kopia |
| wartość ujemna | wszystkie | 8.3 | stężenie poniżej zera jest fizycznie niemożliwe |
| zero pyłu | PM2.5, PM10 | 8.4 | 0 µg/m³ pyłu jest nierealne; zera gazów zostają, bo to najpewniej wartości poniżej progu wykrywalności |
| PM2.5 stałe >= 12 h | PM2.5 1g | 8.5 | zawieszony czujnik: cały ciąg identycznych wartości od 12 godzin |
| PM2.5 jednogodzinny pik | PM2.5 1g | 8.6 | wartość o ponad 50 µg/m³ wyższa od obu sąsiednich godzin; szum, którego pogoda nie wyjaśni |

Przed regułami poprawiamy kody: spację na końcu kodu stacji (8.17) i dwa kody stacji niezgodne ze słownikiem (8.10). Ciągi i piki liczymy na pełnej siatce godzin, więc brak pomiaru przerywa ciąg.

In [2]:
flagged = silver['flag_measurements'](measurements)

(
    flagged
    .group_by('powod_usuniecia')
    .agg(pl.len().alias('wiersze'))
    .with_columns((100 * pl.col('wiersze') / flagged.height).round(3).alias('procent'))
    .sort('wiersze', descending=True)
)

powod_usuniecia,wiersze,procent
str,u32,f64
null,8472637,94.638
"""brak pomiaru""",470793,5.259
"""duplikat stanowiska""",7415,0.083
"""zero pyłu""",1719,0.019
"""PM2.5 jednogodzinny pik""",91,0.001
"""PM2.5 stałe >= 12 h""",69,0.001
"""wartość ujemna""",2,0.0


Usunięte wiersze według wskaźnika (bez braków pomiaru, które dotyczą wszystkich wskaźników):

In [3]:
(
    flagged
    .filter(pl.col('powod_usuniecia').is_not_null() & (pl.col('powod_usuniecia') != 'brak pomiaru'))
    .group_by('powod_usuniecia', 'wskaznik', 'czas_usredniania')
    .agg(pl.len().alias('wiersze'), pl.col('kod_stanowiska').n_unique().alias('stanowiska'))
    .sort('powod_usuniecia', 'wiersze', descending=[False, True])
)

powod_usuniecia,wskaznik,czas_usredniania,wiersze,stanowiska
str,str,str,u32,u32
"""PM2.5 jednogodzinny pik""","""PM2.5""","""1g""",91,30
"""PM2.5 stałe >= 12 h""","""PM2.5""","""1g""",69,4
"""duplikat stanowiska""","""C6H6""","""1g""",7415,1
"""wartość ujemna""","""OC(PM2.5)""","""24g""",2,1
"""zero pyłu""","""PM10""","""1g""",990,12
"""zero pyłu""","""PM2.5""","""1g""",728,6
"""zero pyłu""","""PM10""","""24g""",1,1


## 3. Co zostaje mimo anomalii

- **Wartości skrajne (8.8)** - epizody smogowe 8-10.01.2024 i epizod SO2 w Szczecinie 27.02.2024 są potwierdzone komunikatami GIOŚ. To prawdziwe stężenia, które model powinien widzieć.
- **PM2.5 większe od PM10 i NO2 większe od NOx (8.7)** - różnice najwyżej kilku µg/m³, w granicach niepewności pomiaru.
- **Zera gazów (8.4)** - NO, SO2 i benzen przy bardzo niskich stężeniach.
- **Pomiary po dacie zamknięcia stanowiska (8.13)** - przyrządy stacji mobilnych mierzyły dłużej, niż podaje słownik; wartości wyglądają na prawdziwe.

## 4. Czas w UTC i poprawka kierunku wiatru

GIOŚ podaje czas UTC+1 i znakuje godzinę jej końcem, a Open-Meteo podaje UTC i znakuje godzinę jej początkiem (8.9). Nowa kolumna `data_czas_utc` = `data_czas` − 2 h dla pomiarów godzinowych, więc obie tabele mają tę samą etykietę dla tej samej godziny. Pomiary dobowe (`24g`) mają znacznik doby (00:00) i zostają bez przesunięcia.

W pogodzie przy prędkości wiatru 0 kierunek zmieniamy z 270° na `NULL`, bo przy ciszy jest nieokreślony (8.14).

In [4]:
clean = silver['clean_measurements'](flagged)
weather_clean = silver['clean_weather'](weather)

(
    clean
    .filter(pl.col('kod_stanowiska') == 'DsWrocAlWisn-PM2.5-1g')
    .select('kod_stanowiska', 'data_czas', 'data_czas_utc', 'wartosc')
    .head(3)
)

kod_stanowiska,data_czas,data_czas_utc,wartosc
str,datetime[μs],datetime[μs],f64
"""DsWrocAlWisn-PM2.5-1g""",2024-01-01 01:00:00,2023-12-31 23:00:00,94.0
"""DsWrocAlWisn-PM2.5-1g""",2024-01-01 02:00:00,2024-01-01 00:00:00,44.8
"""DsWrocAlWisn-PM2.5-1g""",2024-01-01 03:00:00,2024-01-01 01:00:00,20.2


## 5. Kontrola wyniku

In [5]:
pm25 = clean.filter((pl.col('wskaznik') == 'PM2.5') & (pl.col('czas_usredniania') == '1g'))

assert clean['wartosc'].null_count() == 0
assert clean.select(pl.struct('kod_stanowiska', 'data_czas').is_duplicated().any()).item() is False
assert clean['wartosc'].min() >= 0
assert clean.filter(pl.col('wskaznik').is_in(['PM2.5', 'PM10']) & (pl.col('wartosc') == 0)).height == 0
assert clean['data_czas_utc'].max().year == 2024
assert clean.filter(pl.col('kod_stacji') != pl.col('kod_stacji').str.strip_chars()).height == 0
assert clean.filter(pl.col('kod_stacji').is_in(list(silver['STATION_CODES']))).height == 0
assert weather_clean.filter((pl.col('wind_speed_10m') == 0) & pl.col('wind_direction_10m').is_not_null()).height == 0

print('pomiary:', flagged.height, '->', clean.height, f'(usunięto {flagged.height - clean.height})')
print('PM2.5 1g:', pm25.height, 'pomiarów na', pm25['kod_stacji'].n_unique(), 'stacjach')

pomiary: 8952726 -> 8472637 (usunięto 480089)
PM2.5 1g: 816760 pomiarów na 96 stacjach


## 6. Zapis do silver

In [6]:
SILVER_DIR.mkdir(parents=True, exist_ok=True)

for name, table in [('pomiary_2024', clean), ('openmeteo_2024', weather_clean)]:
    path = SILVER_DIR / f'{name}.parquet'
    table.write_parquet(path)
    print(f'{path.name:24} {table.height:>10,} wierszy  {path.stat().st_size / 1_000_000:5.1f} MB')

pomiary_2024.parquet      8,472,637 wierszy   28.7 MB


openmeteo_2024.parquet      843,264 wierszy    7.6 MB
